# 📊 ETL — Zona de Aterrizaje (Data Lake) en MongoDB

**Arquitectura del proyecto:**

| Capa | Tecnología | Colección | Contenido |
|---|---|---|---|
| 🥉 **Bronze** (Data Lake) | MongoDB Atlas | `estudiantes_raw` | Datos crudos tal como vienen del CSV (todo como texto) |
| 🥈 **Silver** (limpieza) | MongoDB Atlas | `estudiantes_limpios` | Datos limpios, tipados y validados |
| 🥇 **Gold** (Data Warehouse) | PostgreSQL | *(siguiente etapa)* | Modelo analítico para alerta temprana de deserción |

**Pipeline de este notebook:**
1. Cargar el CSV crudo → colección **Bronze** en MongoDB (sin tocar nada).
2. Leer desde Bronze y hacer el **diagnóstico de calidad** de los datos.
3. **Limpiar** cada columna con reglas explícitas y trazables.
4. Guardar el resultado tipado en la colección **Silver**.

## 1 · Configuración y conexión a MongoDB

> Las credenciales se leen del archivo `.env` (para no dejarlas hardcodeadas en el código que se sube a git).

In [12]:
# Dependencias del proyecto: en la raíz ejecutar ->  pip install -r requirements.txt

In [13]:
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

# Raíz del proyecto (funciona ejecutando desde /notebooks o desde la raíz)
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BASE_DIR))

from src.conn import get_dataset_path, get_mongo_db

pd.set_option("display.max_columns", None)

db = get_mongo_db()
COLECCION_BRONZE = "estudiantes_raw"
COLECCION_SILVER = "estudiantes_limpios"
CSV_PATH = get_dataset_path()

print("Conexión a MongoDB Atlas OK ->", db.client.admin.command("ping"))
print(f"Base: {db.name} | Dataset: {CSV_PATH.name}")


Conexión a MongoDB Atlas OK -> {'ok': 1}
Base: Estudiantes | Dataset: dataset_estudiantes_etl.csv


## 2 · Carga del CSV a la capa 🥉 Bronze

Se Lee el CSV con `dtype=str`: **todo queda como texto**, que es exactamente lo que debe guardar una capa Bronze (los datos crudos, sin transformar). A cada documento le agregamos metadatos de ingesta.

In [14]:
df_raw = pd.read_csv(CSV_PATH, dtype=str)
print(f"Filas del CSV: {len(df_raw):,} | Columnas: {len(df_raw.columns)}")

registros = df_raw.to_dict(orient="records")
for r in registros:
    r["_carga"] = {
        "origen": str(CSV_PATH),
        "fecha_ingesta": datetime.now(timezone.utc).isoformat(),
    }

col_bronze = db[COLECCION_BRONZE]
col_bronze.drop()  # re-carga idempotente: siempre quedan los datos del último run
col_bronze.insert_many(registros, ordered=False)
print(f"🥉 Bronze '{COLECCION_BRONZE}': {col_bronze.count_documents({}):,} documentos insertados")


Filas del CSV: 5,000 | Columnas: 13
🥉 Bronze 'estudiantes_raw': 5,000 documentos insertados


## 3 · Diagnóstico de calidad

Leemos de vuelta desde Bronze (como haría cualquier proceso aguas abajo) y cuantificamos los problemas de cada columna.

In [15]:
datos_bronze = list(col_bronze.find({}, {"_id": 0, "_carga": 0}))
df = pd.DataFrame(datos_bronze)
print(f"Documentos leídos de Bronze: {len(df):,}")
df.head()

Documentos leídos de Bronze: 5,000


,id_alumno,edad_alumno,carrera,cohorte,codigo_materia,nombre_materia,anio_cursada,periodo_inscripcion,tipo_instancia,fecha_examen,nota,asistencias,clases_totales
0,A0164,23,Ingeniería en Sistemas,2018,alg-01,Algoritmos,1,2024-1,Parcial 2,2023-11-06,9,8.0,20
1,A0056,31,Ingeniería en Sistemas,2020,PL-01,Programación Lógica,1,2023-1,TP Práctico,2024-09-25,8,16.0,40
2,A0041,21,Ingeniería en Sistemas,2023,FIS-01,Física,1,2024-1,Final,2023-08-07,7,13.0,20
3,A0012,29,Ingeniería en Sistemas,2020,pl-01,Programación Lógica,1,2023-1,Parcial 1,21/11/2025,10,14.0,30
4,A0059,30,Ingeniería en Sistemas,2023,SO-01,Sistemas Operativos,3,2026-1,Recuperatorio,2023-10-25,6,11.0,20


In [16]:
from src.limpieza import diagnosticar_calidad

datos_bronze = list(col_bronze.find({}, {"_id": 0, "_carga": 0}))
df = pd.DataFrame(datos_bronze)
print(f"Documentos leídos de Bronze: {len(df):,}")

diagnostico = diagnosticar_calidad(df)
print("Problemas de calidad detectados en la capa Bronze:")
display(diagnostico)


Documentos leídos de Bronze: 5,000
Problemas de calidad detectados en la capa Bronze:


,cantidad
codigo_materia · variantes (deben ser 6),24
codigo_materia · formato no normalizado,0
nota · vacíos,0
nota · escritas con palabras (Diez/Ocho/Ausente),156
nota · fuera de rango 0-10,241
edad · vacíos,0
edad · texto ('veinte'),37
edad · outliers (fuera de 16-100),113
asistencias · vacías,235
asistencias · negativas,242


> 🔎 **Detalle clave detectado:** el formato abreviado es **DD-MM-AA** (día-mes-año): el primer campo llega hasta 31 y el segundo solo toma `01–12`. `"21/11/2025"` y `"16-06-24"` lo confirman. El parser unifica los tres formatos a fecha real.

## 4 · Limpieza → capa 🥈 Silver

Las reglas viven en **`src/limpieza.py`** (no en la celda) para poder testearlas
sin Jupyter. Los tests están en `tests/test_limpieza.py`.

Reglas (acordadas para el proyecto):
- **Valores imposibles / fuera de rango → `null`** (no inventamos datos).
- **Palabras → número** cuando hay traducción unívoca (`Diez` → 10).
- **Ausente en examen** → no hay nota: `null`.
- **`codigo_materia`** → formato canónico `XXX-NN` (6 materias reales).
- **Fechas** → unificadas a un solo tipo fecha.
- **Derivado útil para deserción:** `pct_asistencia` %.


In [17]:
from src.limpieza import limpiar_dataframe

df_l, reporte = limpiar_dataframe(df)

print(f"Limpieza aplicada sobre {reporte['filas']:,} filas.")
print("Nulos resultantes por columna:")
display(pd.DataFrame({"nulos": pd.Series(reporte["nulos"])}))


Limpieza aplicada sobre 5,000 filas.
Nulos resultantes por columna:


,nulos
id_alumno,0
edad_alumno,113
carrera,0
cohorte,0
codigo_materia,0
nombre_materia,0
anio_cursada,0
periodo_inscripcion,0
tipo_instancia,0
fecha_examen,0


In [18]:
# ---- Validación post-limpieza (la misma que corren los tests) ----
chequeos = pd.Series(
    {
        "materias únicas (debe ser 6)": reporte["materias_unicas"],
        "fechas sin parsear (debe ser 0)": reporte["fechas_sin_parsear"],
        "notas fuera de 0-10 (debe ser 0)": reporte["notas_fuera_rango"],
        "edades fuera de 16-100 (debe ser 0)": reporte["edades_fuera_rango"],
        "asistencias negativas o mayores a clases (debe ser 0)": reporte["asistencias_inconsistentes"],
        "rango de fechas": reporte["rango_fechas"],
    },
    name="valor",
)
display(chequeos.to_frame())

assert reporte["fechas_sin_parsear"] == 0
assert reporte["notas_fuera_rango"] == 0
assert reporte["edades_fuera_rango"] == 0
assert reporte["asistencias_inconsistentes"] == 0
print("✅ Validación superada: la capa Silver no tiene valores imposibles.")


,valor
materias únicas (debe ser 6),6
fechas sin parsear (debe ser 0),0
notas fuera de 0-10 (debe ser 0),0
edades fuera de 16-100 (debe ser 0),0
asistencias negativas o mayores a clases (debe ser 0),0
rango de fechas,2023-03-01 -> 2025-11-25


✅ Validación superada: la capa Silver no tiene valores imposibles.


## 5 · Carga a la colección 🥈 Silver

Convertimos los tipos de pandas a tipos nativos (BSON): `Int64` → `int|null`, `float` → `double|null`, fecha → `date`.

In [19]:
def doc_silver(fila: "pd.Series") -> dict:
    """Convierte una fila de pandas a un documento BSON-compatible."""
    doc = {}
    for k, v in fila.items():
        try:
            es_nulo = pd.isna(v)
        except (TypeError, ValueError):
            es_nulo = False
        if es_nulo:
            doc[k] = None                      # NaN / NaT / pd.NA -> null en Mongo
        elif isinstance(v, pd.Timestamp):
            doc[k] = v.to_pydatetime()
        elif isinstance(v, (np.integer,)):
            doc[k] = int(v)
        elif isinstance(v, (np.floating,)):
            doc[k] = float(v)
        else:
            doc[k] = v
    return doc


docs_silver = [doc_silver(f) for _, f in df_l.iterrows()]

col_silver = db[COLECCION_SILVER]
col_silver.drop()  # re-carga idempotente
col_silver.insert_many(docs_silver, ordered=False)

# Índices que van a hacer falta en la etapa Gold y en las consultas analíticas
col_silver.create_index("id_alumno")
col_silver.create_index("codigo_materia")
col_silver.create_index("fecha_examen")

print(f"🥈 Silver '{COLECCION_SILVER}': {col_silver.count_documents({}):,} documentos")


🥈 Silver 'estudiantes_limpios': 5,000 documentos


In [20]:
# Muestra de como queda un documento real en MongoDB
list(col_silver.find({"nota": None}).limit(2))

[{'_id': ObjectId('6af77f75dad5d2d18694cef6'),
  'id_alumno': 'A0029',
  'edad_alumno': 21,
  'carrera': 'Ingeniería en Sistemas',
  'cohorte': 2023,
  'codigo_materia': 'BD-03',
  'nombre_materia': 'Base de Datos III',
  'anio_cursada': 4,
  'periodo_inscripcion': '2026-1',
  'tipo_instancia': 'TP Práctico',
  'fecha_examen': datetime.datetime(2025, 1, 25, 0, 0),
  'nota': None,
  'asistencias': 32,
  'clases_totales': 40,
  'pct_asistencia': 80.0},
 {'_id': ObjectId('6af77f75dad5d2d18694cef9'),
  'id_alumno': 'A0196',
  'edad_alumno': 22,
  'carrera': 'Ingeniería en Sistemas',
  'cohorte': 2026,
  'codigo_materia': 'PL-01',
  'nombre_materia': 'Programación Lógica',
  'anio_cursada': 1,
  'periodo_inscripcion': '2023-1',
  'tipo_instancia': 'Recuperatorio',
  'fecha_examen': datetime.datetime(2024, 11, 3, 0, 0),
  'nota': None,
  'asistencias': 19,
  'clases_totales': 30,
  'pct_asistencia': 63.3}]

## 6 · Resumen final de la limpieza

In [21]:
tabla = pd.DataFrame({
    "columna": df.columns,
    "nulos_antes": [int((df[c].str.strip().eq("")).sum()) for c in df.columns],
    "nulos_despues": [int(df_l[c].isna().sum()) for c in df.columns],
    "unicos_antes": [int(df[c].nunique()) for c in df.columns],
    "unicos_despues": [int(df_l[c].nunique()) for c in df.columns],
})
print(f"Filas Bronce leídas: {len(df):,} | Filas Silver guardadas: {len(df_l):,} (no se eliminó ninguna)")
tabla

Filas Bronce leídas: 5,000 | Filas Silver guardadas: 5,000 (no se eliminó ninguna)


,columna,nulos_antes,nulos_despues,unicos_antes,unicos_despues
0,id_alumno,0,0,200,200
1,edad_alumno,0,113,21,17
2,carrera,0,0,2,2
3,cohorte,0,0,9,9
4,codigo_materia,0,0,24,6
5,nombre_materia,0,0,6,6
6,anio_cursada,0,0,4,4
7,periodo_inscripcion,0,0,7,7
8,tipo_instancia,0,0,5,5
9,fecha_examen,0,0,2446,998


### ✅ Resultado

- **🥉 Bronze** `Estudiantes.estudiantes_raw`: 5.000 documentos crudos (todo texto + metadatos de ingesta).
- **🥈 Silver** `Estudiantes.estudiantes_limpios`: 5.000 documentos limpios y tipados, con índices sobre `id_alumno`, `codigo_materia` y `fecha_examen`.
- Los valores imposibles quedaron como `null` (≈600 registros de nota / edad / asistencia), a resolver en la capa Gold.

**Siguiente etapa (Gold / PostgreSQL):** definir el modelo dimensional (ej. `dim_alumno`, `dim_materia`, `dim_tiempo`, `hecho_rendimiento`), Constraints de `NOT NULL`/`CHECK`, y el ETL Mongo → Postgres (pymongo → `psycopgCOPY` o CSV staging).